# Homework 4: **MyTorch**, a tiny automatic differentiation system built on top of NumPy

# CS328 — Numerical Methods for Visual Computing and Machine Learning
- - -


<div style="text-align: center">
<br>
<b>This assignment will be graded automatically through our grading tool.</b>
<br><br>
<b>Due: 17.12.2025 at 21:00 (UTC+1) </b> <br><br> </div>
</div>

This notebook contains literate code, i.e. brief fragments of Python surrounded by descriptive text. Please complete/extend this notebook for your homework submission:

* Only write your solution code in the areas that start with "`# YOUR CODE HERE`". Please do not remove assignments that initialize a variable with an *ellipsis* (`var_name = ...`). Instead, replace the `...` part with the answer. You can add more cells to test your answer.
  
* We use cell tags to prevent you from editing or deleting cells required for our grading tool. If you are using Jupyter Lab / Notebook, you do not need to worry about this, as long as you do not try to delete/change the content of cells that Jupyter Lab/Notebook does not let you edit by default. If you are using some other IDE (VS Code, PyCharm, etc), **you have full responsibility to make sure that the cell tags are not modified, and that you do not change the code of cells with tag `"editable": false`** . In particular, you should not delete any of the existing cells from the Notebook. 
  
* Before handing in, **please make sure that your notebook runs correctly from top to bottom by restarting the kernel and running all cells in order**. Note that one of the read-only cells that is provided in the handout is expected to raise an error. **All other cells should run without errors**. Submissions that cannot be auto-graded because the notebook does not run will receive zero points on failing parts.

Make sure to use the reference Python distribution so that project files can be opened by the TAs. In this course, we use <a href="https://www.anaconda.com/products/individual">Anaconda</a>, specifically the version based on Python 3.13.

<div class="alert alert-warning">
Homework assignments in CS328 count towards your final grade and therefore must be done individually.
</div>

> © Realistic Graphics Lab - EPFL (2025). This notebook is property of the Realistic Graphics Lab at EPFL and may not be redistributed. Posting of notebooks and/or solutions on the web (e.g. on GitHub) is not permitted.

# Overview

In this homework assignment, you will complete a partially finished implementation of a tiny *automatic differentiation* (AD) framework named *MyTorch* and use it to train a neural network so that it can solve a common benchmark problem with a reasonable level of accuracy.

<p style="text-align:center">
<img src="https://rgl.s3.eu-central-1.amazonaws.com/media/uploads/wjakob/2023/11/17/mytorch.png" style="width:150px; height:auto">
</p>

MyTorch uses a *tape*-based differentiation strategy, which we have discussed in Lecture 8 of CS328. This means that it records every  mathematical operation of a program into a data structure known as a *tape* or *computation graph*. Differentiating individual operations (e.g. addition) on this tape is relatively straightforward, and by repeatedly using the *chain rule*, we can then assemble all of these per-operation derivatives into one of the *entire program*.

Suppose that this program is represented by a function $f$, which maps an m-dimensional input $\mathbf{x}$ onto an n-dimensional output $\mathbf{y}$. In that case, the *Jacobian matrix* $\mathbf{J}_f$ is an $n\times m$ matrix containing partial derivatives of every output with respect to every input.

Computing or storing the matrix $\mathbf{J}_f$ is usually extremely expensive and can even be infeasible when both $n$ and $m$ are  large. The surprising property of AD is that it can compute matrix-vector products involving $\mathbf{J}_f$ at a computational cost that is not much greater than that of the original calculation. AD systems usually provide two *modes*:

- **Forward mode**. Given a perturbation $\boldsymbol{\delta}\mathbf{x}$ of the function *inputs*, we seek the resulting perturbation $\boldsymbol{\delta}\mathbf{y}$ of the function *outputs*. It is given by the Jacobian-vector product  $\boldsymbol{\delta}\mathbf{y}=\mathbf{J}_f\boldsymbol{\delta}\mathbf{x}$ and can be computed by replaying the tape from the *start* to the *end*.

- **Reverse/Backward mode**. Given a perturbation $\boldsymbol{\delta}\mathbf{y}$ of the function *outputs*, we seek the resulting perturbation $\boldsymbol{\delta}\mathbf{x}$ of the function *inputs*. It is given by the vector-Jacobian product  $\boldsymbol{\delta}\mathbf{x}=\boldsymbol{\delta}\mathbf{y}^T\mathbf{J}_f$ and can be computed by replaying the tape in reverse from the *end* to the *start*. In the machine learning world, this is called *backpropagation*.

Both can be extremely useful–which one you need really depends on the context. In this assignment, we will first start with forward derivatives and then turn to backward differentiation in the second part.

# Prelude

Let's begin by importing essential NumPy/SciPy/Matplotlib components that are needed to complete this exercise. The following also installs a convenient package named [`tqdm`](https://github.com/tqdm/tqdm) that can visualize the progress of long-running computations. You can ignore potential messages about the need to restart the notebook kernel (this is not actually needed.)

In [ ]:
%pip install tqdm
import numpy as np 
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from IPython import display
import pickle
%matplotlib inline
%config InlineBackend.figure_format='retina'

## A Differentiable `array` Type

The remainder of this notebook will make heavy use of a custom NumPy array class that we will simply call `array`. The main difference between an ordinary NumPy array and this "homework 4 `array`" is that the latter always stores 32-bit floating point numbers, and that it has a `.grad` field that we will use to record the *gradients* associated with each entry of the array. (As explained in Lecture 8 of CS328, we will use the terms *derivative* and *gradient* somewhat loosely since the quantities being differentiated are usually vector-valued.) This gradient is initially zero. To catch mistakes, the implementation refuses to assign gradients, whose shape does not match the original array. You do not need to understand the details of the `array` class implementation, which uses object-oriented Python syntax that was not covered in C328.

In [ ]:
class array(np.ndarray):
    def __new__(cls, value):
        return np.asarray(value, dtype=np.float32).view(cls)

    @property
    def grad(self):
        'Gradient getter. Returns a zero-valued array when a gradient has not yet been assigned.'
        g = getattr(self, '_grad', None)
        if g is None:
            g = np.zeros(self.shape, dtype=np.float32)
        return g

    @grad.setter
    def grad(self, value):
        'Gradient setter. Raises an exception when the user attempts to assign a gradient with an incorrect shape'
        value = np.asarray(value, dtype=np.float32)
        if self.shape != value.shape:
            raise Exception(f'Bad shape: {self.shape} vs. {value.shape}!')
        self._grad = value

The following cell shows an example of how the `array` type can be used. (There is no need to change anything here.)

In [ ]:
x = array([1,2,3])
print(f'x={x} and x.grad={x.grad}')

# Let's assign a gradient
x.grad += [4, 5, 6]
print(f'x={x} and x.grad={x.grad}')

# (Commented) This operation would fail due to a shape conflict (2 vs 3 entries)
# x.grad = [6, 2]

## The Tape

The next cell defines a global variable named `tape` to which operations will be appended. Please do not manipulate this variable directly—our provided code will fill and empty it as needed. The cell also creates two so-called *decorators* ([explanation](https://realpython.com/primer-on-python-decorators/)), which are functions that *modify* other functions. Their role is as follows:

- `@mytorch`:  Prefixing a function with this decorator causes it to add an entry to the tape every time the function is called.
- `@mytorch_fwd`: Add this decorator to provide the forward derivative of a previously defined function. We will use this shortly.

Both access a global dictionary named `ops`—again, please don't modify this dictionary yourself.
The details of what is happening inside these two decorators is not so important, and you may treat them as black boxes.

In [ ]:
tape = []
ops = {}

def mytorch(func):
    def wrapper(*args):
        args = tuple((v if isinstance(v, array) else array(v)) for v in args)
        out = func(*args)
        tape.append((func.__name__, args, out))
        return out
    ops[func.__name__] = wrapper
    return wrapper

def mytorch_fwd(func):
    if func.__name__ not in ops:
        raise RuntimeError(f'Cannot register forward derivative for an unknown function "{func.__name__}"!')
    ops[func.__name__ + '_fwd'] = func
    return ops[func.__name__]

As mentioned above, prefixing a function definition with `@mytorch` will cause it to append a new entry to the tape every time the function is called. The tape stores a sequence of 3-tuples `(name, args, out)`, in which

- `name` refers to the name of the function that created the entry
- `args` lists the function arguments as a tuple
- `out` stores the function return value.

Let's try this. The example tape recording session below defines an addition operation named `add`, runs it once, and then inspects the tape. 

In [ ]:
@mytorch
def add(a, b):
    return a + b

# Try using this function with some constant inputs
add(3, 4)

# Investigate the contents of the tape
print(f'Tape contents: {tape}')

# Let's clear the tape following this experiment
tape.clear()

You should be able to see the three components of the resulting tape entry explained above.

# Section 1: Forward differentiation

We also provide the `forward()` function, which implements the high-level structure of forward-mode AD. Do not change this function. It visits each operation on the tape and raises an exception if the user (i.e., you) did not define a matching forward derivative. If all is well, it executes the forward derivative function and accumulates the resulting gradient into the result of the operation. That's all—the high level structure of such a tape-based AD system is surprisingly short.

In [ ]:
def forward(verbose=True):
    try:
        # Invoke the forward derivative of each operation on the tape
        for (name, args, out) in tape:
            if verbose:
                print(f'forward(): visiting "{name}"')

            # Complain if the forward derivative wasn't defined
            func = ops.get(name + '_fwd', None)
            if func is None:
                raise RuntimeError(f'Forward derivative of "{name}" is not implemented!')

            # Accumulate the computed gradient into the output variable
            out.grad += func(*args)
    finally:
        tape.clear() # Clear the tape at the end (even if an error occurred)

The main challenge will be to provide all needed derivatives. Let's first see what happens when they are missing.

In [ ]:
# Commented-out for submission as it was raising an error due to missing forward derivative for 'add'
# a = array(1)
# b = array(2)
# c = add(a, b)
# forward()

The above lines at first generate an error (this is expected and will not be a problem for grading). To fix it, we will need to implement the missing *forward derivative* of the `add` operation. This is a function that

1. is decorated by `@mytorch_fwd` instead of `@mytorch`,
2. has the same name, 
3. has same arguments, and
4. should compute the derivative of the output based on the function values and the `.grad` field of each input.

The addition operation is defined by the following function
$$
f(a, b):=a+b.
$$
The derivative of this function with respect to its inputs are given by
$$
\frac{\partial f}{\partial a}=1,\quad\text{and}\quad\frac{\partial f}{\partial b}=1.
$$

However, this isn't the whole story because $a$ and $b$ are themselves intermediate results in a longer calculation involving derivatives. In this case, the [chain rule](https://en.wikipedia.org/wiki/Chain_rule) tells us how the full derivative $\mathrm{d}f$ of the operation can be determined based on the above *partial* derivatives and derivatives $\mathrm{d}a$ and $\mathrm{d}b$ associated with the two inputs:
$$
\mathrm{d}f = \frac{\partial f}{\partial a}\,\mathrm{d}a+\frac{\partial f}{\partial b}\,\mathrm{d}b
$$

This means that we simply add up the `.grad` fields of the two inputs (at least for the `add` operation..). Below is what this looks like together with an example usage. Before calling `forward()`, this example code also initializes the `.grad` fields (otherwise `c.grad` will always be zero).

In [ ]:
@mytorch_fwd
def add(a, b):
    return a.grad + b.grad

a = array(1)
b = array(2)
c = add(a, b)

a.grad = 5
b.grad = 7
forward()

print(f'c={c}, c.grad={c.grad}')

As you can see, the error message is gone now. These operations right-multiply the Jacobian matrix of the `add` operation with the input derivative `[5, 7]` which yields the expected result `12.0`.

### A word of caution

The plan for the remainder of this section is as follows: you will add number of additional `@mytorch` and `@mytorch_fwd`-annotated functions that track derivatives `neg`, `mul`, `div`, `exp`, `log`, and so on. It will then be possible to build larger differentiable programs from these operations. In doing so, please pay attention to the following points:

- To perform a differentiable computation, you *must* use the `@mytorch`-decorated functions. For example, `a+b*c` would be written as `add(a, mul(b, c))`. Using normal NumPy operations will *not work* since they aren't recorded onto the tape. Similarly, operations like *indexing into an array* (`a[i]`) won't be captured by our rudimentary tape implementation. Using such steps in a calculation will likely lead to missing or incorrect gradients. 

- When you implement the derivatives themselves (i.e., the code that goes into the `@mytorch_fwd`-decorated function definitions), you should use ordinary NumPy operations and *never* use `@mytorch`-decorated differentiable operations. Otherwise, traversing the tape will *add more entries to it*—the program would get stuck in an infinite loop and eventually run out of memory.

- All of the derivatives in this notebook (forward derivatives in this section, and backward derivatives in the next sections) are **one-liners**. If you find that you need to write a lot of code, then there is likely an issue with your approach.

Popular machine learning frameworks like [PyTorch](https://pytorch.org/tutorials/beginner/basics/intro.html), [TensorFlow](https://www.tensorflow.org/tutorials), or [JAX](https://jax.readthedocs.io/en/latest/) provide types with [overloaded operators](https://realpython.com/operator-function-overloading/) to create a natural interface and avoid the subtleties/inconveniences explained above. However, such a more advanced interface is beyond the scope this CS328 homework exercise.

With that advice, let's go through these operations one by one. Testing is left as an exercise to the reader, we only include one for the first cell. (You don't *have to test*, but it's probably a good idea to spot-check your implementation with an expected input/output pair based on what you learned in MATH-101).

### Negation and subtraction (5pts)

In [ ]:
@mytorch
def neg(a):
    return -a

@mytorch_fwd
def neg(a):
    return -a.grad
    
# Subtraction operation implemented using the already existing
# 'add()' and 'neg()' -> no need to provide a forward derivative
def sub(a, b):
    return add(a, neg(b))



Test it:

In [ ]:
a, b = array(1), array(2)
c = sub(a, b)
a.grad, b.grad = 5, 7
forward(verbose=False)
assert c == -1 and c.grad == -2

### Multiplication (5pts)

In [ ]:
@mytorch
def mul(a, b):
    return a * b

@mytorch_fwd
def mul(a, b):
    return a.grad * b + a * b.grad


In [ ]:
# DO NOT DELETE THIS CELL

### Division (5pts)

In [ ]:
@mytorch
def div(a, b):
    return a / b

@mytorch_fwd
def div(a, b):
    # YOUR CODE HERE
    return (a.grad * b - a * b.grad )/ b**2
    

In [ ]:
# DO NOT DELETE THIS CELL

### The exponential function (5pts)

In [ ]:
@mytorch
def exp(a):
    return np.exp(a)

@mytorch_fwd
def exp(a):
    # YOUR CODE HERE
    return np.exp(a) * a.grad


In [ ]:
# DO NOT DELETE THIS CELL

## Differentiating the Normal Distribution (15pts)
We now have all of the pieces needed to implement a differentiable version of the normal distribution density function
$$
f_N(x, \mu, \sigma)=\frac{1}{\sigma \sqrt{2\pi}} \exp^{-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2}
$$

Complete the function prototype below so that it realizes all parts of the equation using `@mytorch`-decorated differentiable operations. It's fine to create local variables for intermediate results. We did not create an operation for `sqrt()`, but do you really need it?

In [ ]:
def fN(x, mu, sigma):
    # YOUR CODE HERE
    return mul(div(1,mul(sigma,np.sqrt(2*np.pi))),exp(div(neg(mul(div(sub(x, mu), sigma),div(sub(x, mu), sigma))), 2)))


In [ ]:
# DO NOT DELETE THIS CELL

If all above works, then the following should plot the exponential density for $\mu=2$ and $\sigma=3$ on the interval $[0,4]$ along with the derivative with respect to the $\mu$ and $\sigma$ parameters.

In [ ]:
x = array(np.linspace(0,4,1000))
mu, sigma = array(2), array(3)
y1 = fN(x, mu, sigma)
mu.grad, sigma.grad = 1, 0
sigma.grad = 0
forward()
y2 = fN(x, mu, sigma)
mu.grad, sigma.grad = 0, 1
forward()

plt.plot(x, y1, label=r'$f_N$')
plt.plot(x, y1.grad, label=r'$\partial f_N/\partial\mu$')
plt.plot(x, y2.grad, label=r'$\partial f_N/\partial\sigma$')
plt.legend()

The code above demonstrated what *forward mode* is good at: it can quickly provide many output derivatives (in our case, the 1000 function plot values) with respect to a single input or a linear combination of inputs. 

Let's now turn to the *reverse mode*, which is vastly more efficient in the *opposite case* where derivatives with respect to many inputs are desired.

# Section 2: Reverse mode

Reverse mode works in an extremely similar way, except that everything is flipped around. Differentiation traverses the tape from end to start, and it runs a *backward* derivative for each mathematical operation. The backward derivative takes the `.grad` field of the originally computed output and then propagates it to the function inputs. We need to define a new *decorator* to declare these backward derivatives:

In [ ]:
def mytorch_bwd(func):
    if func.__name__ not in ops:
        raise RuntimeError(f'Cannot register backward derivative for an unknown function "{func.__name__}"!')
    ops[func.__name__ + '_bwd'] = func
    return ops[func.__name__]

The *function signature* of a backwards derivative is slightly different:

- It receives all the original function arguments (e.g. ``a``, ``b``) followed by the originally computed function output (`out`).
- The `out` parameter carries gradients that should be forwarded to the inputs. To do so, the function must return a tuple containing the per-input gradients resulting from this operation.

We provide an example of this pattern for the addition operation. Since the per argument-partial derivatives of addition are equal to `1`, the function simply copies the output gradient and forwards it to the inputs:

In [ ]:
@mytorch_bwd
def add(a, b, out):
    return out.grad, out.grad

### Backward derivatives of elementary arithmetic operations (10pts)

Can you analogously define backward derivatives for negation and multiplication?

In [ ]:
@mytorch_bwd
def neg(a, out):
    return -out.grad


@mytorch_bwd
def mul(a, b, out):
    # YOUR CODE HERE
    return out.grad * b, out.grad * a

In [ ]:
# DO NOT DELETE THIS CELL

We also provide one more big chunk of code for you: `backward()`, the entry point for the reverse-mode traversal of the tape. It is almost identical to `forward()` except for two things:

1. Everything is reversed.

2. The additional `unbroadcast()` addresses a technical issue that is unique to reverse mode.

   *In case you are curious about the details*: when NumPy performs calculations, it sometimes *broadcasts* arrays by expanding their size or dimensionality (this is the reason why operations like adding `np.array([1])` to `np.array([1,2,3])` work without problems). While that is of course very convenient, this extra step performs "secret" operations that are now missing from MyTorch's tape. The `unbroadcast()` function implements the backward derivative of such broadcasting operations so that everything is again 100% consistent.

In [ ]:
def unbroadcast(grad, shape):
    return grad.sum(axis=tuple(range(len(shape), len(grad.shape)))) \
               .sum(axis=tuple(i for i, s in enumerate(shape) if s == 1), keepdims=True)

def backward(verbose=True):
    try:
        # Invoke the backward derivative of each operation on the reversed tape
        for (name, args, out) in reversed(tape):
            if verbose:
                print(f'backward(): visiting "{name}"')

            func = ops.get(name + '_bwd', None)
            if func is None:
                raise RuntimeError(f'Backward derivative of "{name}" is not implemented!')

            grad_in = func(*args, out)
            
            # Turn the output into a tuple if it isn't one already
            if not isinstance(grad_in, tuple):
                grad_in = (grad_in, )

            assert len(grad_in) == len(args) # Must return a gradient *per* input
            for a, grad in zip(args, grad_in):
                # Accumulate the computed gradient into the input variable(s)
                a.grad = a.grad + unbroadcast(grad, a.shape)
    finally:
        tape.clear() # Clear the tape at the end (even if an error occurred)

With this function and the backwards derivatives you added above, we can now differentiate a simple calculation.

Can you get the following test to pass?

In [ ]:
a, b, c = array(2), array(3), array(4)
d = sub(a, mul(b, c))
d.grad = 1
backward(d)
assert d == -10 and a.grad == 1 and b.grad == -4 and c.grad == -3

### Backward derivative of division, exponential, and logarithm (15pts)

Let's do a few more operations from before as well as a new one, the *logarithm* (you don't need to implement its forward derivative). 

*An optional suggestion*: to check the correctness of division and exponential, you could see if the behavior of the previously defined function $f_N$ is consistent when comparing forward/backward derivatives.

In [ ]:
@mytorch_bwd
def div(a, b, out):
    # YOUR CODE HERE
    return out.grad/b ,-(a * out.grad )/ b**2

@mytorch_bwd
def exp(a, out):
    # YOUR CODE HERE
    return out.grad * out

@mytorch
def log(x):
    return np.log(x)

@mytorch_bwd
def log(a, out):
    # YOUR CODE HERE
    return out.grad / a

In [ ]:
# DO NOT DELETE THIS CELL

## Remaining bits (15pts)

The final steps of this section should bring the system into a state where it can be used to train a neural network. Right now, we are still missing several operations that are a prerequisite for this:

1. **The `ReLU` activation function**: neural nets require a nonlinear component to be nontrivial. This function provides that source of nonlinearity.
2. **Sum reduction**: This operation sums all components of an input vector and outputs a single value. Derivatives also propagate through this step, hence we must provide its backward derivative.
3. **Matrix multiplication**: this is the computational workhorse of *multi-layer perceptrons* (MLPs).

Let's go through these in the order listed above. You may find the function `np.where` ([documentation](https://numpy.org/doc/stable/reference/generated/numpy.where.html)) helpful.

In [ ]:
@mytorch
def relu(x):
    return np.maximum(x, 0)

@mytorch_bwd
def relu(a, out):
    # YOUR CODE HERE
    return out.grad * np.where(a > 0, 1.0, 0.0)


In [ ]:
# DO NOT DELETE THIS CELL

The sum reduction is the first operation in this notebook where the input and output shape don't agree. The backward derivative of this operation should produce a result matching the shape of the input variable.

In [ ]:
@mytorch
def sum_reduce(x):
    return x.sum()

@mytorch_bwd
def sum_reduce(a, out):
    # YOUR CODE HERE
    return out.grad * np.ones_like(a)

In [ ]:
# DO NOT DELETE THIS CELL

Matrix multiplication is trickiest to get right—at a high level, it resembles scalar multiplication, so some form of the product rule should be used. However, the first problem is that matrices are *non-commutative* under multiplication, so the order in which the operands are written matters a great deal. Second, because reverse-mode derivatives implement a left-multiplication by the Jacobian (or equivalently, a matrix-vector multiplication with the *transpose* of the Jacobian), it means that some of the
operands must now be transposed.

You may find the following report by Mike Giles to be a helpful resource ([LINK](https://people.maths.ox.ac.uk/gilesm/files/NA-08-01.pdf)). Section 2.2.2 on page 4 specifically covers the case of matrix multiplication.

In [ ]:
@mytorch
def matmul(a, b):
    return a @ b

@mytorch_bwd
def matmul(a, b, out):
    # YOUR CODE HERE
    return out.grad @ b.T, a.T @ out.grad

In [ ]:
# DO NOT DELETE THIS CELL

# Section 3: Using MyTorch to train a multi-layer perceptron

With all of these pieces in place, we now have an AD system that can backpropagate gradients through a neural network. We will use it to train a *multi-layer perceptron* that can classify different types of clothing from [Fashion-MNIST](https://en.wikipedia.org/wiki/Fashion_MNIST) benchmark dataset released by Zalando. This dataset consists of 60'000 *training images* of clothing articles belonging to 10 different classes, along with another 10'000 *testing images* to assess the accuracy of trained models.

Begin by downloading the file `fashion_mnist.pickle` from [this link](https://d38rqfq1h7iukm.cloudfront.net/media/uploads/wjakob/2023/11/17/fashion_mnist.pickle). Copy this file into the same directory that also contains the Jupyter notebook. The cell below loads and plots a tiny subset of this dataset so that we can get a feeling for what's inside.

In [ ]:
with open('fashion_mnist.pickle', 'rb') as f:
    dataset = pickle.load(f)
    train_images = dataset['train_images']
    train_labels = dataset['train_labels']
    test_images = dataset['test_images']
    test_labels = dataset['test_labels']
    classes = dataset['classes']
    
def plot_image_matrix(data, data_classes, data_classes_pred=None):
    figure, ax = plt.subplots(5,12, figsize=(12,5))
    for i in range(5*12):
        ax.ravel()[i].imshow(data[i], cmap='gray_r')
        ax.ravel()[i].set_axis_off()
        color = 'black'
        if data_classes_pred is not None:
            color = 'red' if data_classes[i] != data_classes_pred[i] else 'green'
        ax.ravel()[i].set_title(classes[data_classes[i]], y=-.4, color=color)
    plt.tight_layout()

plot_image_matrix(train_images, train_labels)

## Network architecture

Ensure that you have either attended or watched the video of lecture 9, otherwise the following may not make much sense.

Each fashion-MNIST images consists of 28$\times$28=784 grayscale pixels. We will pass these through the following sequence of operations:

<img src="https://rgl.s3.eu-central-1.amazonaws.com/media/uploads/wjakob/2023/11/17/fashion-mnist.svg" width="100%">

1. reshaping the `28x28` input into a `784x1` column vector. The operation `np.reshape` ([documentation](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html#)) may be useful for this.
2. Re-scaling the `0-255`-valued `uint8` image into a floating point image with pixel values in the range `[0, 1]`.
3. Processing the image by a dense layer (weight multiplication + bias) that takes the  `784x1` vector and outputs a `512x1` vector that is subsequently transformed by a `ReLU` activation function.
4. Processing that vector using a second dense layer with `512` neurons, which outputs another `512x1` vector and applying the `ReLU` activation function once more.
5. Processing the vector by a last dense layer, which reduces the input to a `10x1` -shaped column vector.
6. Instead of `ReLU`, the last layer applies the task-specific `SoftMax` activation function, which is more suitable for applications that require choosing one of multiple categories.

The neural calculation is controlled by entries of the `params` dictionary that we will shortly define.

### `SoftMax` activation (5pts)

For an $n$-dimensional input vector $\mathbf{x}$, the `SoftMax` activation function is defined as
$$
\texttt{SoftMax}(\mathbf{x})_i=\frac{\exp(x_i)}{\sum_{j=1}^n\exp(x_j)}
$$
Use previously defined `@mytorch`-decorated functions to realize it below.

In [ ]:
def softmax(x):
    # YOUR CODE HERE
    return div(exp(x),sum_reduce(exp(x)))

In [ ]:
# DO NOT DELETE THIS CELL

The cell below creates the initial *weights* (`w0`, `w1`, `w2`) and *biases*  (`b0`, `b1`, `b2`) of the dense layers.  You may treat this code as a black box and don't need to look into the details or change anything here. *In case you are interested to learn more*: There is a whole science to randomly initializing neural networks in a *good way*—the following uses a method known as *He initialization* (introduced in the linked [paper](https://arxiv.org/abs/1502.01852)). It initializes the weight matrices with normally distributed variates having a carefully chosen standard deviation related to the number of layer inputs. Bias vectors remain zero-initialized.

In [ ]:
def random_array(shape):
    if shape[-1] == 1:
        result = np.zeros(shape)
    else:
        result = np.random.randn(*shape) * np.sqrt(2/shape[-1])
    return array(result)

# Initialize the random number generator with a fixed
# state so that reuse of this notebook remains deterministic
np.random.seed(7)

# Create initial neural network parameters for 3 linear layers
params = {
    'w0' : random_array((512, 784)),
    'b0' : random_array((512, 1)),
    'w1' : random_array((512, 512)),
    'b1' : random_array((512, 1)),
    'w2' : random_array((10, 512)),
    'b2' : random_array((10, 1))
}

### Model definition (15pts)

Complete the `model(image)` function below using `@mytorch`-decorated functions so that it implements the sequence of steps explained and illustrated at the beginning of this section. To access the weight matrix and bias vector of the *first* layer (index 0), write `params['w0']` and `params['b0']`, and so on. 

One point is worth highlighting: the `matmul()` operation you defined earlier realizes a differentiable version of *matrix-matrix* multiplication, but it likely won't work for *matrix-vector* multiplication (matrices and vectors have a different dimension, which presents a tricky special case). The solution is easy: we will represent vectors as $N\times 1$ matrices ("*column vectors*"), in which case matrix-vector and matrix-matrix multiplication are literally the same thing.

In [ ]:
def model(image):
    # YOUR CODE HERE
    x_raw = array(image.reshape(-1, 1))
    x = div(x_raw, 255.0) 

    h0_linear = add(matmul(params['w0'], x), params['b0'])
    h0 = relu(h0_linear)
    
    h1_linear = add(matmul(params['w1'], h0), params['b1'])
    h1 = relu(h1_linear)
    
    out_linear = add(matmul(params['w2'], h1), params['b2'])
    output = softmax(out_linear)
    
    return output


Let's try running this (untrained) model on an image just to see that this works and produces a 10D vector.

In [ ]:
assert model(train_images[0]).shape == (10, 1)

In [ ]:
# DO NOT DELETE THIS CELL

The initial model weights were randomly initialized, so we don't expect this initial to perform particularly well. The following cell visualizes the predictions for the first 60 training images. A *green* label indicates that the model predicted the right category, while red indicates that it is wrong. Most of them are wrong as you can see, and the accuracy is on the order of 10-11% (i.e., just random guessing.)

In [ ]:
def test_accuracy(model):
    num_correct = 0
    for i in range(len(test_images)):
        v = model(test_images[i]).argmax()
        tape.clear()
        # We aren't doing any training here, clear the tape so that it doesn't grow to a huge size
        if v == test_labels[i]:
            num_correct += 1
    return num_correct / len(test_images) * 100

plot_image_matrix(test_images, [model(img).argmax() for img in test_images[:5*12]], test_labels)

print(f'Accuracy of untrained model on test dataset : {test_accuracy(model):.2f} %.')

Before we can improve the quality of this initial model, we must first quantify how good it actually is, by applying a *loss function* to the predictions. 

The choice of loss function is highly related to the structure of the network. In our case, the last layer applied a `SoftMax` activation to generate a *probability vector* (i.e. a vector, whose entries sum to one). The visualization above used the largest element of this vector to associate each output with one of the 10 types of clothing articles.

However, things aren't quite so binary -- the model can be *more or less confident* in its predictions, and the loss function should not only penalize wrong answers but also correct answers that lack confidence (the network output associated with the correct class should be close to `1.0`).

The standard choice in this specific scenerio is the *sparse cross entropy* loss (you can read more about it [here](https://www.pinecone.io/learn/cross-entropy-loss/)). The *cross entropy* measures the difference between two probability vectors $\mathbf{t}$ and $\mathbf{p}$ and is defined as

## Sparse cross-entropy (5pts)
$$
H(\mathbf{t}, \mathbf{p})= -\sum_{i=1}^n t_i\log p_i 
$$
In our case, $\mathbf{p}$ is the neural network output, and $\mathbf{t}$ is the *target probability distribution*. The word *sparse* means that the target probability vector is "*one-hot*" (i.e. it consists of all zeros except for one component that is turned on).
$$
t_i=\begin{cases}
1,\text{if the answer has class $i$,}\\
0,\text{otherwise.}
\end{cases}
$$
Implement this sparse cross-entropy loss below using `@mytorch`-decorated operations for steps of this calculation that must be differentiable. It should take a column vector of type `array` as the `value` parameter and an `int` in the range `0...len(value)-1` indicating class membership.

In [ ]:
def sparse_cross_entropy(value, category):
    # YOUR CODE HERE
    target_vector_np = np.zeros((10, 1), dtype=np.float32)
    target_vector_np[category, 0] = 1.0
    one_hot_target_t = array(target_vector_np.T) 
    p_category = matmul(one_hot_target_t, value)
    log_p = log(p_category)
    
    return sum_reduce(neg(log_p))

In [ ]:
sparse_cross_entropy(array(np.ones((10, 1))*0.1), 1)

In [ ]:
assert sparse_cross_entropy(array(np.ones((10, 1))*0.1), 1) > 0

That's it, congratulations for making it all the way to here. If your code works as as instructed, you should be able to just run the next cell and enjoy the rest of the show.

Here is what will happen: our code will repeatedly call the model, loss function, and then back-propagate gradients to the elements of the `params` dictionary. 
The training loop looks as follows

```python
step_size = 0.001
for i in range(len(train_images)):
    loss = sparse_cross_entropy(model(train_images[i]), train_labels[i])

    loss.grad = 1
    backward(verbose=False) # Backpropagate!

    for k, v in params.items():
        params[k] = array(v - v.grad * step_size) # Gradient descent
```
Every iteration takes a gradient descent step using the most basic form of *Stochastic Gradient Descent* (SGD) with a step size of `0.001`. The next cell is quite a bit longer than the snippet above, and that is because it also adds a nice progress bar and a visualization of the decay of averaged losses.

In [ ]:
# This cell creates a dynamic plot and updates it progressively. This
# has been tested to work in Jupyter lab. Your mileage may vary if you
# use another editor like VS Code.

step_size = 0.001

fig, ax = plt.subplots()

plot_frequency = 100
loss_x, loss_y = [], []
plot = ax.plot(loss_x,loss_y)[0]
ax.set_ylim(0, 2.6)
dh = display.display(fig, display_id=True)
progress = tqdm(range(len(train_images)))
loss_avg = 0
tape.clear()
if 'no_train' in params:
    progress = []

for i in progress:
    loss = sparse_cross_entropy(model(train_images[i]), train_labels[i])
    
    # Backpropagate!
    loss.grad = 1
    backward(verbose=False)
    
    # Keep a running average of the last 100 loss values and plot them
    loss_avg += loss
    if i % plot_frequency == plot_frequency - 1:
        loss_x.append(len(loss_x))
        loss_y.append(loss_avg / plot_frequency)
        plot.set_xdata(loss_x)
        plot.set_ydata(loss_y)
        ax.set_xlim(0, len(loss_x))
        ax.autoscale()
        fig.canvas.draw()
        dh.update(fig)
        progress.set_description(f'Avg. loss value = {loss_avg/plot_frequency:,.02f}')
        loss_avg = 0

    # Run gradient descent
    for k, v in params.items():
        params[k] = array(v - v.grad * step_size)
plt.close()

Let's re-run the accuracy test and visualization of model predictions after training.

In [ ]:
print(f'Accuracy on test dataset (after training)  : {test_accuracy(model):.2f} %.')

In [ ]:
plot_image_matrix(test_images, [model(img).argmax() for img in test_images[:5*12]], test_labels)

A few more comments: training for 1 *epoch* takes 8 minutes on my machine (a 2021 M1 Macbook) and improves the prediction accuracy on the test set from 10% to roughly 84%. Because of differences across machines, operating systems, and Python/NumPy versions, you may not achieve these exact numbers. But if you observe large discrepancies, this could be indicative of a bug or inefficiencies in your code. Be aware that we will *not* run the Fashion-MNIST training during auto-grading, so it's not a big problem if your code runs slowly (instead, we will test specific MyTorch operations in a more targeted manner).

While training, you may observe that the loss plot decreases rapidly and then starts to stagnate. It still decreases slightly albeit very slowly, and training for a *second epoch* (you can re-run the training loop as many times as you want) further improves prediction accuracy to ~86% on my machine. This is quite respectable such a simple implementation, though a number of improvements can push accuracy beyond 90% and accelerate training significantly. The following improvements would be considered "low-hanging fruit":

- We're using simple gradient descent. A better optimizer (gradient descent with momentum or [Adam](https://arxiv.org/abs/1412.6980)) enables faster and more robust convergence essentially "for free".
- The architecture network is very much on the simple end and lacks many architectural finesses (*convolutions*, *dropout layers*, a higher number of hidden layers, etc).
- Our `backward()` function propagates derivatives *everywhere* (to constants and even the *input image*). Mature ML training systems track what actually needs derivatives and then only compute this subset.
- We're separately evaluating the network for each image. In practice, it is usually more efficient to perform several network evaluations at the same time and then average the resulting gradients, which is called a *mini-batch*.
- In recent years, *graphics cards* have proven to be a superior match for the highly regular operations (matrix multiplications, convolutions) involved in training neural networks. Depending on the workload, a $\sim$10-20$\times$ speedup obtained by simply by running the computation on the GPU instead of the CPU is not uncommon.

These are just some idea for extensions in case you found this homework intriguing and wish to further extend it. However, please don't submit such extensions on Moodle as they will likely confuse our auto-grading scripts 😉.